# 🌫️ AQI Predictor — Islamabad
### End-to-end ML Pipeline
Run cells in order. Restart runtime after Cell 1.

In [ ]:
# Cell 1: Install all dependencies
!pip install "hopsworks[python]==4.7.*" pandas numpy requests tqdm confluent-kafka scikit-learn shap matplotlib joblib hops-deltalake flask pyngrok

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 761.9/761.9 kB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.5/57.5 MB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2/124.2 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.2/44.2 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 98.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 94.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 73.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 258.6/258.6 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.2/295.2 kB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.6/140.6 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.3/45.3 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.9/14.9 MB 60.7 MB/s eta 0:00:00
   ━━━━━━━━

⚠️ **Restart runtime after Cell 1, then continue from Cell 2**

In [ ]:
# Cell 2: Set your API keys (replace with your actual keys)
import os
os.environ["AQICN_TOKEN"]       = "7667197b43e0afb1dc370a41c435269919569e62"
os.environ["HOPSWORKS_API_KEY"] = "Vwtdg7Cq2Fpfqbzd.o2snX82DVNiucmBTQv7q0zLtZA4YMdIu2G7CswR2jgTx7K8nmHPsXcvPkP99r7ON"
os.environ["CITY"]              = "islamabad"
print("✅ Keys set!")

✅ Keys set!


In [ ]:
import requests
import pandas as pd
import numpy as np
import hopsworks
from tqdm import tqdm
import os

AQICN_TOKEN   = os.environ["AQICN_TOKEN"]
HOPSWORKS_KEY = os.environ["HOPSWORKS_API_KEY"]

# Fetch real current AQI
resp = requests.get(f"https://api.waqi.info/feed/islamabad/?token={AQICN_TOKEN}", timeout=10).json()
real_aqi = float(resp["data"]["aqi"]) if resp["status"] == "ok" else 100.0
print(f"✅ Real AQI: {real_aqi}")

# Fetch only 1 month of weather (much faster!)
url = (
    "https://archive-api.open-meteo.com/v1/archive?"
    "latitude=33.72148&longitude=73.04329"
    "&start_date=2024-12-01&end_date=2024-12-31"
    "&hourly=temperature_2m,relativehumidity_2m,windspeed_10m,precipitation"
    "&timezone=UTC"
)
data = requests.get(url, timeout=30).json()
df = pd.DataFrame({
    "timestamp":     pd.to_datetime(data["hourly"]["time"]),
    "temperature":   data["hourly"]["temperature_2m"],
    "humidity":      [int(x) for x in data["hourly"]["relativehumidity_2m"]],
    "windspeed":     data["hourly"]["windspeed_10m"],
    "precipitation": data["hourly"]["precipitation"],
})

df["city"]            = "islamabad"
df["aqi"]             = real_aqi
df["hour"]            = df["timestamp"].dt.hour.astype("int32")
df["day"]             = df["timestamp"].dt.day.astype("int32")
df["month"]           = df["timestamp"].dt.month.astype("int32")
df["weekday"]         = df["timestamp"].dt.weekday.astype("int32")
df["is_weekend"]      = (df["weekday"] >= 5).astype("int32")
df["hour_sin"]        = np.sin(2 * np.pi * df["hour"] / 24)
df["hour_cos"]        = np.cos(2 * np.pi * df["hour"] / 24)
df["month_sin"]       = np.sin(2 * np.pi * df["month"] / 12)
df["month_cos"]       = np.cos(2 * np.pi * df["month"] / 12)
df["aqi_lag_1h"]      = real_aqi
df["aqi_lag_3h"]      = real_aqi
df["aqi_lag_6h"]      = real_aqi
df["aqi_lag_24h"]     = real_aqi
df["aqi_change_rate"] = 0.0
df["aqi_rolling_6h"]  = real_aqi
df["aqi_rolling_24h"] = real_aqi

print(f"Built {len(df)} rows")

# Push to Hopsworks
project = hopsworks.login(api_key_value=HOPSWORKS_KEY)
fs = project.get_feature_store()
fg = fs.get_or_create_feature_group(
    name="aqi_features",
    version=1,
    primary_key=["timestamp", "city"],
    description="AQI features for Islamabad",
    event_time="timestamp",
)
batch_size = 500
for i in tqdm(range(0, len(df), batch_size), desc="Uploading"):
    fg.insert(df.iloc[i:i+batch_size], write_options={"wait_for_job": False})
print(f"✅ Uploaded {len(df)} rows!")

✅ Real AQI: 154.0
Built 744 rows

Logged in to project, explore it here https://eu-west.cloud.hopsworks.ai:443/p/31999


Uploading:   0%|          | 0/2 [00:00<?, ?it/s]

Feature Group created successfully, explore it at 
https://eu-west.cloud.hopsworks.ai:443/p/31999/fs/20683/fg/38622


Uploading: 100%|██████████| 2/2 [00:54<00:00, 27.46s/it]

✅ Uploaded 744 rows!


In [ ]:
# Cell 3: Backfill historical data into Hopsworks Feature Store
import requests
import pandas as pd
import numpy as np
from datetime import datetime
import hopsworks
from tqdm import tqdm
import os

AQICN_TOKEN   = os.environ["AQICN_TOKEN"]
HOPSWORKS_KEY = os.environ["HOPSWORKS_API_KEY"]

# Fetch real current AQI
resp = requests.get(f"https://api.waqi.info/feed/islamabad/?token={AQICN_TOKEN}", timeout=10).json()
real_aqi = float(resp["data"]["aqi"]) if resp["status"] == "ok" else 100.0
print(f"✅ Real AQI for Islamabad: {real_aqi}")

# Fetch 2 years of historical weather
url = (
    "https://archive-api.open-meteo.com/v1/archive?"
    "latitude=33.72148&longitude=73.04329"
    "&start_date=2023-01-01&end_date=2024-12-31"
    "&hourly=temperature_2m,relativehumidity_2m,windspeed_10m,precipitation"
    "&timezone=UTC"
)
data = requests.get(url, timeout=30).json()
df = pd.DataFrame({
    "timestamp":     pd.to_datetime(data["hourly"]["time"]),
    "temperature":   data["hourly"]["temperature_2m"],
    "humidity":      [int(x) for x in data["hourly"]["relativehumidity_2m"]],
    "windspeed":     data["hourly"]["windspeed_10m"],
    "precipitation": data["hourly"]["precipitation"],
})

# Build features
df["city"]            = "islamabad"
df["aqi"]             = real_aqi
df["hour"]            = df["timestamp"].dt.hour.astype("int32")
df["day"]             = df["timestamp"].dt.day.astype("int32")
df["month"]           = df["timestamp"].dt.month.astype("int32")
df["weekday"]         = df["timestamp"].dt.weekday.astype("int32")
df["is_weekend"]      = (df["weekday"] >= 5).astype("int32")
df["hour_sin"]        = np.sin(2 * np.pi * df["hour"] / 24)
df["hour_cos"]        = np.cos(2 * np.pi * df["hour"] / 24)
df["month_sin"]       = np.sin(2 * np.pi * df["month"] / 12)
df["month_cos"]       = np.cos(2 * np.pi * df["month"] / 12)
df["aqi_lag_1h"]      = real_aqi
df["aqi_lag_3h"]      = real_aqi
df["aqi_lag_6h"]      = real_aqi
df["aqi_lag_24h"]     = real_aqi
df["aqi_change_rate"] = 0.0
df["aqi_rolling_6h"]  = real_aqi
df["aqi_rolling_24h"] = real_aqi

print(f"Built {len(df)} rows of training data.")

# Push to Hopsworks Feature Store
project = hopsworks.login(api_key_value=HOPSWORKS_KEY)
fs = project.get_feature_store()
fg = fs.get_or_create_feature_group(
    name="aqi_features",
    version=2,
    primary_key=["timestamp", "city"],
    description="Hourly AQI features with real AQICN data",
    event_time="timestamp",
)
batch_size = 500
for i in tqdm(range(0, len(df), batch_size), desc="Uploading"):
    fg.insert(df.iloc[i:i+batch_size], write_options={"wait_for_job": False})
print(f"✅ Uploaded {len(df)} historical rows to Feature Store!")

✅ Real AQI for Islamabad: 154.0
Built 17544 rows of training data.

Logged in to project, explore it here https://eu-west.cloud.hopsworks.ai:443/p/31999


Uploading:   0%|          | 0/36 [00:00<?, ?it/s]

Feature Group created successfully, explore it at 
https://eu-west.cloud.hopsworks.ai:443/p/31999/fs/20683/fg/38559


Uploading: 100%|██████████| 36/36 [2:58:54<00:00, 298.19s/it]

✅ Uploaded 17544 historical rows to Feature Store!


In [ ]:
# Cell 4: Train ML model and save to Model Registry
import os, hopsworks, pandas as pd, numpy as np, joblib, json
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

HOPSWORKS_KEY = os.environ["HOPSWORKS_API_KEY"]

# Generate realistic Islamabad AQI training data
np.random.seed(42)
n = 5000
hours    = np.random.randint(0, 24, n)
days     = np.random.randint(1, 32, n)
months   = np.random.randint(1, 13, n)
weekdays = np.random.randint(0, 7, n)
base_temp    = 15 + 15 * np.sin((months - 3) * np.pi / 6)
temperature  = base_temp + np.random.normal(0, 3, n)
humidity     = 40 + 20 * np.random.random(n)
windspeed    = 5 + 10 * np.random.random(n)
precipitation= np.random.exponential(0.5, n)
base_aqi     = 100 + 60 * np.sin((months - 10) * np.pi / 6)
hour_effect  = 20 * np.sin((hours - 8) * np.pi / 12)
wind_effect  = -2 * windspeed
aqi = np.clip(base_aqi + hour_effect + wind_effect + np.random.normal(0, 15, n), 30, 400)

df = pd.DataFrame({
    "temperature":    temperature,
    "humidity":       humidity,
    "windspeed":      windspeed,
    "precipitation":  precipitation,
    "hour":           hours,
    "day":            days,
    "month":          months,
    "weekday":        weekdays,
    "is_weekend":     (weekdays >= 5).astype(int),
    "hour_sin":       np.sin(2 * np.pi * hours / 24),
    "hour_cos":       np.cos(2 * np.pi * hours / 24),
    "month_sin":      np.sin(2 * np.pi * months / 12),
    "month_cos":      np.cos(2 * np.pi * months / 12),
    "aqi_lag_1h":     aqi + np.random.normal(0, 5, n),
    "aqi_lag_3h":     aqi + np.random.normal(0, 8, n),
    "aqi_lag_6h":     aqi + np.random.normal(0, 10, n),
    "aqi_lag_24h":    aqi + np.random.normal(0, 15, n),
    "aqi_change_rate":np.random.normal(0, 5, n),
    "aqi_rolling_6h": aqi + np.random.normal(0, 5, n),
    "aqi_rolling_24h":aqi + np.random.normal(0, 8, n),
    "aqi":            aqi
})

FEATURE_COLS = [c for c in df.columns if c != "aqi"]
X = df[FEATURE_COLS].values
y = df["aqi"].values
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train all models and pick best
models = {
    "Ridge":             Ridge(alpha=1.0),
    "Random Forest":     RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1),
    "Gradient Boosting": GradientBoostingRegressor(n_estimators=200, learning_rate=0.05, max_depth=5, random_state=42),
}
results = {}
for name, m in models.items():
    m.fit(X_train, y_train)
    preds = m.predict(X_test)
    rmse = float(np.sqrt(mean_squared_error(y_test, preds)))
    mae  = float(mean_absolute_error(y_test, preds))
    r2   = float(r2_score(y_test, preds))
    results[name] = {"model": m, "rmse": rmse, "mae": mae, "r2": r2}
    print(f"{name:25s} → RMSE={rmse:.2f}  MAE={mae:.2f}  R²={r2:.4f}")

best_name = min(results, key=lambda k: results[k]["rmse"])
best = results[best_name]
print(f"\n🏆 Best model: {best_name}")

# Save locally
os.makedirs("models_v5", exist_ok=True)
joblib.dump(best["model"], "models_v5/best_model.pkl")
with open("models_v5/feature_names.json", "w") as f:
    json.dump(FEATURE_COLS, f)
with open("models_v5/metrics.json", "w") as f:
    json.dump({"rmse": best["rmse"], "r2": best["r2"], "mae": best["mae"]}, f)

# Save to Hopsworks Model Registry
project = hopsworks.login(api_key_value=HOPSWORKS_KEY)
mr = project.get_model_registry()
model_meta = mr.sklearn.create_model(
    name="aqi_predictor",
    version=5,
    metrics={"rmse": best["rmse"], "r2": best["r2"]},
    description=f"Best model: {best_name} — trained on realistic Islamabad AQI patterns",
)
model_meta.save("models_v5")
print("✅ Model version 5 saved to Hopsworks Model Registry!")

Ridge                     → RMSE=2.69  MAE=2.17  R²=0.9960
Random Forest             → RMSE=3.00  MAE=2.22  R²=0.9950
Gradient Boosting         → RMSE=2.94  MAE=2.18  R²=0.9952

🏆 Best model: Ridge

Logged in to project, explore it here https://eu-west.cloud.hopsworks.ai:443/p/31999


ModelRegistryException: Model with name aqi_predictor and version 5 already exists, please select another version.

In [ ]:
import hopsworks, joblib, json, os

project = hopsworks.login(api_key_value=os.environ["HOPSWORKS_API_KEY"])
mr = project.get_model_registry()

model_meta = mr.sklearn.create_model(
    name="aqi_predictor",
    version=6,
    metrics={"rmse": 2.69, "r2": 0.9960},
    description="Ridge Regression — best model for Islamabad AQI",
)
model_meta.save("models_v5")
print("✅ Model version 6 saved!")


Logged in to project, explore it here https://eu-west.cloud.hopsworks.ai:443/p/31999


  0%|          | 0/6 [00:00<?, ?it/s]

Uploading /content/models_v5/best_model.pkl: 0.000%|          | 0/713 elapsed<00:00 remaining<?

Uploading /content/models_v5/feature_names.json: 0.000%|          | 0/269 elapsed<00:00 remaining<?

Uploading /content/models_v5/metrics.json: 0.000%|          | 0/79 elapsed<00:00 remaining<?

Model created, explore it at https://eu-west.cloud.hopsworks.ai:443/p/31999/models/aqi_predictor/6
✅ Model version 6 saved!


In [ ]:
# Cell 5: Generate SHAP Feature Importance plot
import shap, joblib, json, numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

model = joblib.load("models_v5/best_model.pkl")
with open("models_v5/feature_names.json") as f:
    feature_names = json.load(f)

X_sample = np.random.rand(100, len(feature_names)) * 100
explainer  = shap.Explainer(model.predict, X_sample)
shap_values = explainer(X_sample)

shap.plots.bar(shap_values, max_display=15, show=False)
plt.tight_layout()
plt.savefig("shap_plot.png", dpi=120, bbox_inches="tight")
plt.close()
print("✅ SHAP plot saved as shap_plot.png")

✅ SHAP plot saved as shap_plot.png


In [ ]:
# Cell 6: Flask REST API — serves AQI predictions
import threading, time, requests, os, joblib, json, numpy as np
from flask import Flask, jsonify
from datetime import datetime

app_flask = Flask(__name__)

model = joblib.load("models_v5/best_model.pkl")
with open("models_v5/feature_names.json") as f:
    feature_names = json.load(f)

def aqi_label(v):
    if v <= 50:   return "Good"
    if v <= 100:  return "Moderate"
    if v <= 150:  return "Unhealthy for Sensitive Groups"
    if v <= 200:  return "Unhealthy"
    if v <= 300:  return "Very Unhealthy"
    return "Hazardous"

def get_weather():
    url = (
        "https://api.open-meteo.com/v1/forecast?"
        "latitude=33.72148&longitude=73.04329"
        "&hourly=temperature_2m,relativehumidity_2m,windspeed_10m,precipitation"
        "&forecast_days=1&timezone=UTC"
    )
    return requests.get(url, timeout=10).json()["hourly"]

@app_flask.route("/")
def home():
    return jsonify({"message": "AQI Predictor API is running!", "endpoints": ["/predict", "/forecast", "/health"]})

@app_flask.route("/health")
def health():
    return jsonify({"status": "ok", "timestamp": datetime.utcnow().isoformat()})

@app_flask.route("/predict")
def predict():
    hourly = get_weather()
    now = datetime.utcnow().hour
    features = {
        "temperature":    hourly["temperature_2m"][now],
        "humidity":       hourly["relativehumidity_2m"][now],
        "windspeed":      hourly["windspeed_10m"][now],
        "precipitation":  hourly["precipitation"][now],
        "hour":           now, "day": datetime.utcnow().day,
        "month":          datetime.utcnow().month,
        "weekday":        datetime.utcnow().weekday(),
        "is_weekend":     int(datetime.utcnow().weekday() >= 5),
        "hour_sin":       np.sin(2 * np.pi * now / 24),
        "hour_cos":       np.cos(2 * np.pi * now / 24),
        "month_sin":      np.sin(2 * np.pi * datetime.utcnow().month / 12),
        "month_cos":      np.cos(2 * np.pi * datetime.utcnow().month / 12),
        "aqi_lag_1h": 100, "aqi_lag_3h": 100, "aqi_lag_6h": 100, "aqi_lag_24h": 100,
        "aqi_change_rate": 0, "aqi_rolling_6h": 100, "aqi_rolling_24h": 100,
    }
    X = np.array([[features.get(f, 0) for f in feature_names]])
    pred = float(np.clip(model.predict(X)[0], 0, 500))
    return jsonify({"city": "islamabad", "predicted_aqi": round(pred, 1),
                    "status": aqi_label(pred), "timestamp": datetime.utcnow().isoformat()})

@app_flask.route("/forecast")
def forecast():
    hourly = get_weather()
    results = []
    for i in range(min(24, len(hourly["time"]))):
        h = i % 24
        features = {
            "temperature":    hourly["temperature_2m"][i],
            "humidity":       hourly["relativehumidity_2m"][i],
            "windspeed":      hourly["windspeed_10m"][i],
            "precipitation":  hourly["precipitation"][i],
            "hour": h, "day": 1, "month": datetime.utcnow().month,
            "weekday": datetime.utcnow().weekday(),
            "is_weekend": int(datetime.utcnow().weekday() >= 5),
            "hour_sin": np.sin(2 * np.pi * h / 24),
            "hour_cos": np.cos(2 * np.pi * h / 24),
            "month_sin": np.sin(2 * np.pi * datetime.utcnow().month / 12),
            "month_cos": np.cos(2 * np.pi * datetime.utcnow().month / 12),
            "aqi_lag_1h": 100, "aqi_lag_3h": 100, "aqi_lag_6h": 100, "aqi_lag_24h": 100,
            "aqi_change_rate": 0, "aqi_rolling_6h": 100, "aqi_rolling_24h": 100,
        }
        X = np.array([[features.get(f, 0) for f in feature_names]])
        pred = float(np.clip(model.predict(X)[0], 0, 500))
        results.append({"time": hourly["time"][i], "predicted_aqi": round(pred, 1), "status": aqi_label(pred)})
    return jsonify({"city": "islamabad", "forecast": results})

# Run Flask in background thread
def run_flask():
    app_flask.run(host="0.0.0.0", port=5000, debug=False, use_reloader=False)

thread = threading.Thread(target=run_flask, daemon=True)
thread.start()
time.sleep(2)

# Test the API
resp = requests.get("http://localhost:5000/predict")
print("Flask API Response:", resp.json())
resp2 = requests.get("http://localhost:5000/health")
print("Health Check:", resp2.json())

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [25/Apr/2026 19:21:29] "GET /predict HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [25/Apr/2026 19:21:29] "GET /health HTTP/1.1" 200 -


Flask API Response: {'city': 'islamabad', 'predicted_aqi': 99.9, 'status': 'Moderate', 'timestamp': '2026-04-25T19:21:29.652915'}
Health Check: {'status': 'ok', 'timestamp': '2026-04-25T19:21:29.659703'}
